# LeetCode Hot 100 - Day 21

## 今日主题：动态规划（二）

今天的四道题都是**面试高频**，而且状态定义各有特点：

1. 零钱兑换：完全背包，求最少硬币数。
2. 最长递增子序列：经典 O(n²) DP，注意状态定义。
3. 单词拆分：字符串上的 DP，用集合加速查找。
4. 乘积最大子数组：同时维护最大值和最小值，加练题。

今天要继续用昨天的四步法，重点看**状态转移方程怎么想出来**。


## 今天怎么学

1. 零钱兑换和完全平方数是同一个套路，可以对比着看。
2. 最长递增子序列是“以 i 结尾”这种状态定义的典型代表。
3. 单词拆分要把“切分点”当作状态转移的入口。
4. 乘积最大子数组的坑在于负负得正，必须同时记录最小乘积。

最低目标：独立写出零钱兑换和最长递增子序列。


## 今日题单

1. 零钱兑换（LeetCode 322，中等，必做）
2. 最长递增子序列（LeetCode 300，中等，必做）
3. 单词拆分（LeetCode 139，中等，必做）
4. 乘积最大子数组（LeetCode 152，中等，加练）


## 昨日复习

先用 `day20_practice.ipynb` 重写：

1. 爬楼梯。
2. 打家劫舍。

口述：打家劫舍的状态转移方程是什么？为什么是 `dp[i-2] + nums[i]`？


## 题目 1 做题前先补：“凑不出来”怎么表示

零钱兑换：给几种硬币面额和一个总金额，求凑出这个金额最少需要几枚硬币。凑不出来返回 -1。

`dp[i]` 的定义：**凑出金额 i 最少需要几枚硬币。**

转移：枚举最后一枚硬币的面额 `coin`（要求 `coin <= i`）：

```text
dp[i] = min(dp[i - coin] + 1)
```

**难点在“凑不出来”怎么表示。** 做法是用一个“不可能取到的大数”当初始值，比如 `amount + 1`（因为答案最多就是全用 1 元硬币，即 amount 枚）。这样：

- 如果最后 `dp[amount]` 还是大于 amount，说明它从来没被更新过，也就是凑不出来，返回 -1；
- 否则返回 `dp[amount]`。

注意初始值 `dp[0] = 0`：凑出 0 元不需要硬币。


In [ ]:
coins = [1, 2, 5]
amount = 6
dp = [amount + 1] * (amount + 1)
dp[0] = 0

for i in range(1, amount + 1):
    for coin in coins:
        if coin <= i:
            candidate = dp[i - coin] + 1
            if candidate < dp[i]:
                dp[i] = candidate
    if dp[i] > amount:
        print("凑出", i, "：暂时不可能")
    else:
        print("凑出", i, "最少需要", dp[i], "枚硬币")
print("最终答案：", dp[amount])


# 题目 1：零钱兑换

LeetCode 322. Coin Change

## 题目描述（改写版）

给你一个硬币面额数组 `coins` 和一个总金额 `amount`，请计算凑出这个金额**最少**需要多少枚硬币。如果任何组合都凑不出来，返回 `-1`。

每种硬币的数量是无限的。

## 输入

- `coins`：长度 1 到 12，面额 1 到 2 的 31 次方减 1。
- `amount`：0 到 10000。

## 输出

返回最少硬币数；凑不出来返回 -1。

## 示例

示例 1：`coins = [1,2,5]`，`amount = 11`，`11 = 5 + 5 + 1`，返回 3。

示例 2：`coins = [2]`，`amount = 3`，凑不出来，返回 -1。

示例 3：`coins = [1]`，`amount = 0`，返回 0。

## 易漏细节

- `amount = 0` 时答案必须是 0。
- 面额可能比 amount 大，要判断 `coin <= i`。
- 用 `amount + 1` 当“无穷大”，最后用它来判断是否凑不出来。


## 解法名称

**完全背包动态规划（Unbounded Knapsack DP）**。

## 暴力思路

递归枚举用几枚哪种硬币，重复子问题极多，指数级。

## 优化思路

- `dp[i]`：凑出金额 i 最少需要几枚；
- 初始值 `dp[0] = 0`，其余全部设成 `amount + 1`；
- 外层遍历金额 `i` 从 1 到 amount，内层遍历硬币；
- 转移：如果 `coin <= i`，则 `dp[i] = min(dp[i], dp[i - coin] + 1)`；
- 最后判断 `dp[amount]` 是否还大于 amount。

时间 O(amount × 硬币数)，空间 O(amount)。

**面试常问的追问**：如果要求“每个面额只能用一次”怎么办？那就是 0/1 背包，外层改成遍历硬币、内层倒序遍历金额。


## 你来写：零钱兑换

要求：

- 用一维 DP 写，初始值用 `amount + 1` 表示不可达。
- `amount = 0` 直接返回 0（其实是循环自然得出的）。
- 写完用示例的三组数据各跑一遍。

先在心里回答：为什么用 `amount + 1` 当“无穷大”是安全的？


In [ ]:
# 题目：零钱兑换
# 解法：完全背包动态规划（Unbounded Knapsack DP）
# 输入：硬币面额数组 coins（长度 1 到 12），目标金额 amount（0 到 10000）。
# 目标：用最少的硬币凑出 amount，硬币可以无限使用。
# 输出：返回最少硬币数；凑不出来返回 -1。
# 注意：dp[0] = 0，其余初始化成 amount + 1；要判断 coin <= i；最后用 dp[amount] > amount 判断不可达。


def coin_change(coins, amount):
    # 在这里写你的代码
    pass


print(coin_change([1, 2, 5], 11))


In [ ]:
print(coin_change([1, 2, 5], 11))    # 期望 3
print(coin_change([2], 3))           # 期望 -1
print(coin_change([1], 0))           # 期望 0
print(coin_change([2, 5], 10))       # 期望 2
print(coin_change([3], 9))           # 期望 3
print(coin_change([5], 3))           # 期望 -1


## 参考答案：零钱兑换

```python
def coin_change_answer(coins, amount):
    dp = [amount + 1] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                candidate = dp[i - coin] + 1
                if candidate < dp[i]:
                    dp[i] = candidate

    if dp[amount] > amount:
        return -1
    return dp[amount]
```

面试表达：

我用完全背包的动态规划，`dp[i]` 表示凑出金额 i 最少需要的硬币数。数组先全部初始化成 `amount + 1`，这是一个不可能达到的值，所以还能区分“凑不出来”的情况；`dp[0]` 设为 0。然后从小到大枚举金额，对每个金额枚举所有硬币面额，如果面额不超过当前金额，就用 `dp[i - 面额] + 1` 去更新最小值。最后如果 `dp[amount]` 还大于 amount，说明它没有被任何组合更新过，返回 -1，否则返回它。时间 O(amount 乘硬币数)，空间 O(amount)。


In [ ]:


def coin_change_answer(coins, amount):
    dp = [amount + 1] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in coins:
            if coin <= i:
                candidate = dp[i - coin] + 1
                if candidate < dp[i]:
                    dp[i] = candidate

    if dp[amount] > amount:
        return -1
    return dp[amount]


print(coin_change_answer([1, 2, 5], 11))    # 3
print(coin_change_answer([2], 3))           # -1
print(coin_change_answer([1], 0))           # 0
print(coin_change_answer([2, 5], 10))       # 2
print(coin_change_answer([5], 3))           # -1


## 题目 2 做题前先补：状态定义为什么是“以 i 结尾”

最长递增子序列（LIS）看起来很简单：找最长的递增子序列。但状态怎么定义？

**错误定义**：`dp[i]` = 前 i 个数里最长递增子序列的长度。
为什么错？因为“前 i 个数的最长递增子序列”可能不以第 i 个数结尾，这样后面想接上去就不知道能不能接。

**正确定义**：`dp[i]` = **以 `nums[i]` 结尾**的最长递增子序列长度。

有了这个定义，转移就清楚了：`dp[i]` 可以接在前面任何一个比它小的数后面：

```text
dp[i] = max(dp[j]) + 1，其中 j < i 且 nums[j] < nums[i]
```

最终答案不是 `dp[-1]`，而是**所有 dp 里的最大值**，因为最长递增子序列可能以任意一个位置结尾。

这个“以 i 结尾”的思路在很多 DP 题里都会用到，是今天的重点。


In [ ]:
nums = [10, 9, 2, 5, 3, 7, 101, 18]
dp = [1] * len(nums)

for i in range(len(nums)):
    for j in range(i):
        if nums[j] < nums[i] and dp[j] + 1 > dp[i]:
            dp[i] = dp[j] + 1
    print("以", nums[i], "结尾的最长递增子序列长度：", dp[i])

print("dp 数组：", dp)
print("答案是其中的最大值：", max(dp))


# 题目 2：最长递增子序列

LeetCode 300. Longest Increasing Subsequence

## 题目描述（改写版）

给你一个整数数组 `nums`，请找出其中**最长严格递增子序列**的长度。

子序列是指从数组里删掉若干元素（也可以不删）后剩下的序列，**不要求连续**。严格递增是指后面的元素必须严格大于前面的。

## 输入

- `nums`：长度 1 到 2500，元素 -10000 到 10000。

## 输出

返回最长递增子序列的长度。

## 示例

示例 1：`nums = [10,9,2,5,3,7,101,18]`，最长递增子序列是 `[2,3,7,101]`，返回 4。

示例 2：`nums = [7,7,7,7]`，严格递增，只能选一个，返回 1。

示例 3：`nums = [0,1,0,3,2,3]`，最长是 `[0,1,2,3]`，返回 4。

## 易漏细节

- 子序列不要求连续，子数组才要求连续。
- 是严格递增，`[7,7]` 不算。
- 答案不是 `dp[-1]`，而是 `dp` 里的最大值。


## 解法名称

**O(n²) 动态规划（LIS DP）**；进阶解法是**贪心 + 二分（Patience Sorting）**。

## 暴力思路

枚举所有子序列，指数级。

## 优化思路

定义 `dp[i]` = 以 `nums[i]` 结尾的最长递增子序列长度。

- 初始化：每个位置单独成序列，长度都是 1；
- 对每个 i，往前看所有 `j < i`，如果 `nums[j] < nums[i]`，就能把 `nums[i]` 接在 `dp[j]` 后面；
- 取所有可行 j 里最大的 `dp[j] + 1`；
- 答案是 `max(dp)`。

时间 O(n²)，空间 O(n)。n 是 2500，完全够用。

**进阶**：用贪心维护一个“递增数组”，每次用二分查找找到替换位置，时间 O(n log n)。面试时说清楚 O(n²) 版本后可以提一句进阶解法。


## 你来写：最长递增子序列

要求：

- `dp[i]` 表示以 `nums[i]` 结尾的最长递增子序列长度。
- 内层往前找 `nums[j] < nums[i]` 的情况。
- 写完用示例的三组数据各跑一遍。

先在心里回答：为什么答案要取 `max(dp)`，而不是 `dp[-1]`？


In [ ]:
# 题目：最长递增子序列
# 解法：O(n²) 动态规划（LIS DP）
# 输入：整数数组 nums，长度 1 到 2500。
# 目标：求最长严格递增子序列的长度（子序列不要求连续）。
# 输出：返回长度（整数）。
# 注意：dp[i] 表示以 nums[i] 结尾的最长长度；初始化全为 1；答案是 max(dp)。


def length_of_lis(nums):
    # 在这里写你的代码
    pass


print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]))


In [ ]:
print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]))    # 期望 4
print(length_of_lis([7, 7, 7, 7]))                    # 期望 1
print(length_of_lis([0, 1, 0, 3, 2, 3]))              # 期望 4
print(length_of_lis([1]))                             # 期望 1
print(length_of_lis([5, 4, 3, 2, 1]))                 # 期望 1
print(length_of_lis([1, 2, 3, 4, 5]))                 # 期望 5


## 参考答案：最长递增子序列

```python
def length_of_lis_answer(nums):
    if len(nums) == 0:
        return 0

    dp = [1] * len(nums)
    best = 1

    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                if dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
        if dp[i] > best:
            best = dp[i]

    return best
```

面试表达：

我定义 `dp[i]` 为以 `nums[i]` 结尾的最长递增子序列长度。每个位置初始化成 1，因为单独一个元素本身就是长度为 1 的递增子序列。然后对每个 i 往前遍历所有 j，只要 `nums[j]` 小于 `nums[i]`，就说明可以把 `nums[i]` 接在以 j 结尾的序列后面，用 `dp[j] + 1` 去更新 `dp[i]` 的最大值。最后答案是所有 `dp` 里的最大值，因为最长序列可能以任意位置结尾。时间 O(n 的平方)，空间 O(n)。如果数据量更大，可以用贪心加二分查找做到 O(n log n)。


In [ ]:


def length_of_lis_answer(nums):
    if len(nums) == 0:
        return 0

    dp = [1] * len(nums)
    best = 1

    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                if dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
        if dp[i] > best:
            best = dp[i]

    return best


print(length_of_lis_answer([10, 9, 2, 5, 3, 7, 101, 18]))    # 4
print(length_of_lis_answer([7, 7, 7, 7]))                    # 1
print(length_of_lis_answer([0, 1, 0, 3, 2, 3]))              # 4
print(length_of_lis_answer([5, 4, 3, 2, 1]))                 # 1
print(length_of_lis_answer([1, 2, 3, 4, 5]))                 # 5


## 题目 3 做题前先补：把字符串切分看成 DP

单词拆分：给一个字符串 s 和一个单词表，判断 s 能不能由单词表里的词拼成（词可以重复用）。

关键想法：**如果 s 的前 j 个字符能拼出来，而且从 j 到 i 这一段正好是一个单词，那么前 i 个字符也能拼出来。**

所以：

- `dp[i]` 表示“s 的前 i 个字符能不能被拼出来”，是布尔值；
- `dp[0] = True`（空串能拼出来，这是递推的起点）；
- 转移：对每个 i，枚举切分点 j（0 到 i-1），如果 `dp[j]` 为真且 `s[j:i]` 在单词表里，那么 `dp[i] = True`。

为什么要把单词表转成 `set`？因为要频繁做 `in` 判断，集合的查找是 O(1)，列表是 O(n)。这是字典/集合最典型的用法之一。

注意切片 `s[j:i]` 是左闭右开，正好取第 j 到第 i-1 个字符，长度是 `i - j`。


In [ ]:
# 手工看一次切分
s = "leetcode"
word_dict = ["leet", "code"]
word_set = set(word_dict)

print("s[0:4] =", s[0:4], "，在单词表里吗：", s[0:4] in word_set)
print("s[4:8] =", s[4:8], "，在单词表里吗：", s[4:8] in word_set)

dp = [False] * (len(s) + 1)
dp[0] = True
for i in range(1, len(s) + 1):
    for j in range(i):
        if dp[j] and s[j:i] in word_set:
            dp[i] = True
            break
print("dp 数组：", dp)
print("能不能拼出来：", dp[len(s)])


# 题目 3：单词拆分

LeetCode 139. Word Break

## 题目描述（改写版）

给你一个字符串 `s` 和一个字符串列表 `wordDict`。请判断 `s` 能否被拆分成一个或多个 `wordDict` 里的单词。同一个单词可以重复使用。

## 输入

- `s`：长度 1 到 300，只含小写字母。
- `wordDict`：长度 1 到 1000，每个单词长度 1 到 20。

## 输出

能拆分返回 `True`，否则返回 `False`。

## 示例

示例 1：`s = "leetcode"`，`wordDict = ["leet","code"]`，返回 `True`。

示例 2：`s = "applepenapple"`，`wordDict = ["apple","pen"]`，返回 `True`（apple 可以重复用）。

示例 3：`s = "catsandog"`，`wordDict = ["cats","dog","sand","and","cat"]`，返回 `False`。

## 易漏细节

- 单词可以重复使用。
- `dp[0] = True` 是递推的起点。
- 单词表要转成集合，否则查找太慢。
- 找到一个可行的切分点就可以 `break`，不用继续找。


## 解法名称

**动态规划 + 哈希集合（DP with Hash Set）**。

## 暴力思路

用回溯枚举所有切法，最坏是指数级。

## 优化思路

- `dp[i]`：s 的前 i 个字符能否拼出来（布尔）；
- `dp[0] = True`；
- 对每个 i 枚举切分点 j，检查 `dp[j] and s[j:i] in word_set`；
- 一旦成立就把 `dp[i]` 设为 True 并跳出内层循环。

时间 O(n²)（每对 (j, i) 检查一次，切片和集合查找平均接近 O(1)），空间 O(n)。

**面试延伸**：如果要求返回所有拆分方案，就把布尔 DP 换成“回溯 + 记忆化”，或者用 DP 记录前驱。


## 你来写：单词拆分

要求：

- 先把 `wordDict` 转成 `set`。
- 用布尔 DP 写，内层遍历切分点 j。
- 写完用示例的三组数据各跑一遍。

先在心里回答：为什么 `dp[0]` 要设成 `True`？


In [ ]:
# 题目：单词拆分
# 解法：动态规划 + 哈希集合（DP with Hash Set）
# 输入：字符串 s（长度 1 到 300），单词列表 wordDict（每个单词长度 1 到 20）。
# 目标：判断 s 能否拆成一个或多个 wordDict 里的单词，单词可重复使用。
# 输出：能拆分返回 True，否则返回 False。
# 注意：dp[i] 表示前 i 个字符能否拼出来；dp[0] = True；单词表转成 set；找到就 break。


def word_break(s, word_dict):
    # 在这里写你的代码
    pass


print(word_break("leetcode", ["leet", "code"]))


In [ ]:
print(word_break("leetcode", ["leet", "code"]))                  # 期望 True
print(word_break("applepenapple", ["apple", "pen"]))            # 期望 True
print(word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]))    # 期望 False
print(word_break("a", ["a"]))                                   # 期望 True
print(word_break("a", ["b"]))                                   # 期望 False
print(word_break("aaaaaaa", ["aaaa", "aaa"]))                   # 期望 True


## 参考答案：单词拆分

```python
def word_break_answer(s, word_dict):
    word_set = set(word_dict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True

    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break

    return dp[n]
```

面试表达：

我用动态规划。`dp[i]` 表示字符串前 i 个字符能不能由字典里的单词拼成，`dp[0]` 是 True，因为空串不需要拼。然后从小到大枚举结尾位置 i，再枚举切分点 j：如果前 j 个字符能拼出来，并且从 j 到 i 这一段正好是字典里的一个单词，那么前 i 个字符也能拼出来，就把 `dp[i]` 设为真并跳出内层循环。为了方便查找，我先把单词列表转成集合，把 `in` 判断降到常数时间。时间 O(n 的平方)，空间 O(n)。


In [ ]:


def word_break_answer(s, word_dict):
    word_set = set(word_dict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True

    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break

    return dp[n]


print(word_break_answer("leetcode", ["leet", "code"]))            # True
print(word_break_answer("applepenapple", ["apple", "pen"]))      # True
print(word_break_answer("catsandog", ["cats", "dog", "sand", "and", "cat"]))   # False
print(word_break_answer("aaaaaaa", ["aaaa", "aaa"]))             # True


## 题目 4 做题前先补：负负得正带来的麻烦

求最大子数组乘积，本来想仿照“最大子数组和”只维护一个最大值，但会遇到一个问题：

**负数！** 一个很小的负数，乘上下一个负数，会立刻变成很大的正数。

举例：`[2, 3, -2, 4]`，如果只看最大值，到 -2 时会放弃前面的乘积；但如果是 `[-2, -3, -4]`，前两个负数相乘得到正 6，这个 6 恰恰来自“最小的那个负数”。

所以必须**同时维护以当前位置结尾的最大乘积和最小乘积**：

```text
新最大 = max(当前数, 之前最大 × 当前数, 之前最小 × 当前数)
新最小 = min(当前数, 之前最大 × 当前数, 之前最小 × 当前数)
```

答案就是遍历过程中出现过的最大乘积。

**这个“同时维护最大和最小”的技巧是本题的核心，面试官最爱问这一点。**


In [ ]:
nums = [2, 3, -2, 4]
max_here = nums[0]
min_here = nums[0]
best = nums[0]

for i in range(1, len(nums)):
    num = nums[i]
    candidates = [num, max_here * num, min_here * num]
    new_max = candidates[0]
    new_min = candidates[0]
    for value in candidates:
        if value > new_max:
            new_max = value
        if value < new_min:
            new_min = value
    max_here = new_max
    min_here = new_min
    if max_here > best:
        best = max_here
    print("到", num, "为止：最大乘积", max_here, "，最小乘积", min_here, "，历史最佳", best)

print("答案：", best)


# 题目 4：乘积最大子数组

LeetCode 152. Maximum Product Subarray

## 题目描述（改写版）

给你一个整数数组 `nums`，请找出其中**乘积最大的连续子数组**，返回这个乘积。

子数组必须是连续的。

## 输入

- `nums`：长度 1 到 20000，元素 -10 到 10。

## 输出

返回最大乘积。

## 示例

示例 1：`nums = [2,3,-2,4]`，最大乘积是 6（子数组 `[2,3]`）。

示例 2：`nums = [-2,0,-1]`，最大乘积是 0。

示例 3：`nums = [-2,3,-4]`，最大乘积是 24（整段相乘）。

## 易漏细节

- 必须同时维护以当前位置结尾的最大值和最小值。
- 初始值用第一个元素，不能设成 0（因为有负数）。
- 答案要一路比较取最大，不能只看最后的状态。


## 解法名称

**动态规划 + 双状态（DP with Max and Min）**。

## 暴力思路

枚举所有子数组，计算乘积，时间 O(n²) 甚至 O(n³)。

## 优化思路

维护两个变量：

- `max_here`：以当前元素结尾的子数组的最大乘积；
- `min_here`：以当前元素结尾的子数组的最小乘积（可能是很大的负数，遇到下一个负数会翻身）。

每一步：

- 候选值有三个：当前元素本身、`max_here * 当前元素`、`min_here * 当前元素`；
- 从候选里取最大和最小，分别作为新的 `max_here` 和 `min_here`；
- 用 `max_here` 更新全局答案。

时间 O(n)，空间 O(1)。


## 你来写：乘积最大子数组

要求：

- 同时维护 `max_here` 和 `min_here`。
- 初始值都是 `nums[0]`。
- 写完用示例的三组数据各跑一遍。

先在心里回答：为什么只维护最大值会出错？


In [ ]:
# 题目：乘积最大子数组
# 解法：动态规划 + 双状态（DP with Max and Min）
# 输入：整数数组 nums，长度 1 到 20000，元素 -10 到 10。
# 目标：求乘积最大的连续子数组的乘积。
# 输出：返回最大乘积（整数）。
# 注意：必须同时维护以当前位置结尾的最大和最小乘积；初始值用 nums[0]；答案一路取最大。


def max_product(nums):
    # 在这里写你的代码
    pass


print(max_product([2, 3, -2, 4]))


In [ ]:
print(max_product([2, 3, -2, 4]))     # 期望 6
print(max_product([-2, 0, -1]))      # 期望 0
print(max_product([-2, 3, -4]))      # 期望 24
print(max_product([2]))              # 期望 2
print(max_product([-2]))             # 期望 -2
print(max_product([0, 2]))           # 期望 2


## 参考答案：乘积最大子数组

```python
def max_product_answer(nums):
    max_here = nums[0]
    min_here = nums[0]
    best = nums[0]

    for i in range(1, len(nums)):
        num = nums[i]
        candidates = [num, max_here * num, min_here * num]

        new_max = candidates[0]
        new_min = candidates[0]
        for value in candidates:
            if value > new_max:
                new_max = value
            if value < new_min:
                new_min = value

        max_here = new_max
        min_here = new_min

        if max_here > best:
            best = max_here

    return best
```

面试表达：

我维护两个变量：以当前元素结尾的最大乘积和最小乘积。为什么要最小值？因为当前元素如果是负数，之前的最小乘积乘以它反而可能变成最大的正数。每一步的候选值是当前元素本身、之前的最大乘积乘以当前元素、之前的最小乘积乘以当前元素，从这三个里取最大和最小，分别更新两个状态，并用最大乘积更新全局答案。所有状态都用第一个元素初始化，避免负数和 0 造成错误。时间 O(n)，空间 O(1)。


In [ ]:


def max_product_answer(nums):
    max_here = nums[0]
    min_here = nums[0]
    best = nums[0]

    for i in range(1, len(nums)):
        num = nums[i]
        candidates = [num, max_here * num, min_here * num]

        new_max = candidates[0]
        new_min = candidates[0]
        for value in candidates:
            if value > new_max:
                new_max = value
            if value < new_min:
                new_min = value

        max_here = new_max
        min_here = new_min

        if max_here > best:
            best = max_here

    return best


print(max_product_answer([2, 3, -2, 4]))     # 6
print(max_product_answer([-2, 0, -1]))       # 0
print(max_product_answer([-2, 3, -4]))       # 24
print(max_product_answer([-2]))              # -2


# 今日小结

今天四个状态定义：

1. **零钱兑换**：`dp[i]` = 凑出金额 i 的最少硬币数，枚举最后一枚硬币；用 `amount + 1` 表示不可达。
2. **最长递增子序列**：`dp[i]` = **以 i 结尾**的最长长度，答案是 `max(dp)`。
3. **单词拆分**：`dp[i]` = 前 i 个字符能否拼成，枚举切分点；单词表转 `set`。
4. **乘积最大子数组**：同时维护以当前位置结尾的最大和最小乘积。

三条经验：

- **状态定义里带上“以 i 结尾”，往往能让转移更简单**；
- **“不可达”要用一个不可能的值表示**（比如 `amount + 1`）；
- **遇到负数、或者“最大/最小”会互相转换时，要多维护一个状态**。


## 今日复盘区

- 零钱兑换里，为什么用 `amount + 1` 而不是 `float("inf")`？
- 最长递增子序列的 `dp[i]` 定义是什么？答案为什么不是 `dp[-1]`？
- 单词拆分里，为什么要把单词表转成集合？
- 乘积最大子数组里，为什么必须同时维护最小乘积？
- 今天哪几道题能不看答案写出来？

完成情况记录：

- 独立写出：
- 卡住的题：
- 明天重写：
- 完成日期：
